# Entertainment RAG on IMDb + Million Song Dataset

This notebook downloads two public datasets, builds a shared retrieval index, and uses a local **Llama 3.2 3B Instruct** model to answer questions from the top-k retrieved documents.

Colab notes:
- Designed to run on a **T4** with 4-bit quantization for the LLM.
- The original Million Song Dataset is huge, so the notebook uses a **Colab-sized metadata slice** from the Hugging Face processed version.
- You can raise the dataset limits later if you have more time or memory.

## 1. Install Dependencies

This notebook uses lightweight retrieval plus a 4-bit Llama model, so it stays practical on a T4 GPU.

In [ ]:
import importlib.util
import subprocess
import sys

requirements = [
    ("accelerate", "accelerate"),
    ("bitsandbytes", "bitsandbytes"),
    ("datasets", "datasets"),
    ("faiss-cpu", "faiss"),
    ("pandas", "pandas"),
    ("sentence-transformers", "sentence_transformers"),
    ("transformers", "transformers"),
    ("huggingface_hub", "huggingface_hub"),
]

missing = [package for package, module in requirements if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Dependencies ready")

## 2. Setup

If you are in Colab, mount Drive for optional caching. The notebook will also read a Hugging Face token from an environment variable or Colab secret when loading Llama 3.2 3B.

In [ ]:
import os
from pathlib import Path

import pandas as pd

try:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
except Exception as exc:
    print(f"Colab Drive mount skipped: {exc}")

WORK_DIR = Path("/content/drive/MyDrive/entertainment_rag") if Path("/content/drive/MyDrive").exists() else Path.cwd() / "entertainment_rag"
WORK_DIR.mkdir(parents=True, exist_ok=True)

IMDB_LIMIT = int(os.environ.get("IMDB_LIMIT", "12000"))
MSD_LIMIT = int(os.environ.get("MSD_LIMIT", "12000"))
TOP_K = int(os.environ.get("TOP_K", "5"))
CHUNK_CHAR_LIMIT = int(os.environ.get("CHUNK_CHAR_LIMIT", "1200"))

HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    or os.environ.get("hf_token")
)

try:
    if not HF_TOKEN:
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or userdata.get("HUGGING_FACE_HUB_TOKEN")
except Exception:
    pass

print(f"Work dir: {WORK_DIR}")
print(f"IMDb limit: {IMDB_LIMIT}")
print(f"Million Song limit: {MSD_LIMIT}")
print(f"Top-k: {TOP_K}")

## 3. Download Datasets and Build Documents

The notebook uses the Hugging Face `imdb` dataset plus a processed Million Song Dataset slice. Full MSD is too large for a T4, so the notebook indexes a manageable metadata subset.

In [ ]:
import math
import re
from typing import Any

import numpy as np
from datasets import load_dataset


def normalize_text(text: Any) -> str:
    return re.sub(r"\s+", " ", str(text or "").replace("\xa0", " ")).strip()


def is_missing(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if value == "":
        return True
    return False


def first_nonempty(row: dict, keys: list[str]) -> str:
    for key in keys:
        value = row.get(key)
        if isinstance(value, list):
            value = ", ".join(str(item) for item in value if not is_missing(item))
        if not is_missing(value):
            return normalize_text(value)
    return ""


def row_to_imdb_doc(row: dict) -> dict:
    sentiment = "positive" if int(row.get("label", 0)) == 1 else "negative"
    review_text = normalize_text(row.get("text", ""))
    review_text = review_text[:CHUNK_CHAR_LIMIT]
    title = f"IMDb review ({sentiment})"
    text = f"Source: IMDb review. Sentiment: {sentiment}. Review: {review_text}"
    return {
        "source": "imdb",
        "title": title,
        "text": text,
        "metadata": {"sentiment": sentiment, "label": int(row.get("label", 0))},
    }


def row_to_music_doc(row: dict) -> dict:
    title = first_nonempty(row, ["song_name", "title", "track_name", "name"])
    artist = first_nonempty(row, ["artist_name", "artist", "artist_names"])
    album = first_nonempty(row, ["release", "album", "album_name"])
    genre = first_nonempty(row, ["genre", "genres", "artist_terms", "tags"])
    year = first_nonempty(row, ["year", "song_year", "release_year"])

    pieces = []
    if title:
        pieces.append(f"Song: {title}")
    if artist:
        pieces.append(f"Artist: {artist}")
    if album:
        pieces.append(f"Album: {album}")
    if year:
        pieces.append(f"Year: {year}")
    if genre:
        pieces.append(f"Tags: {genre}")

    feature_keys = [
        "tempo",
        "loudness",
        "danceability",
        "energy",
        "valence",
        "acousticness",
        "instrumentalness",
        "speechiness",
    ]
    for key in feature_keys:
        value = row.get(key)
        if not is_missing(value):
            pieces.append(f"{key}: {value}")

    if not pieces:
        pieces.append(normalize_text(row))

    text = "Source: Million Song Dataset. " + "; ".join(pieces)
    return {
        "source": "million_song",
        "title": title or artist or "Million Song record",
        "text": text[:CHUNK_CHAR_LIMIT],
        "metadata": {
            "title": title,
            "artist": artist,
            "album": album,
            "genre": genre,
            "year": year,
        },
    }


print("Downloading IMDb slice...")
imdb_ds = load_dataset("imdb", split=f"train[:{IMDB_LIMIT}]")
print("Downloading Million Song Dataset slice...")
msd_ds = load_dataset("RecSysTUM/Million_Song_Dataset", split=f"train[:{MSD_LIMIT}]")

imdb_docs = [row_to_imdb_doc(row) for row in imdb_ds]
msd_docs = [row_to_music_doc(row) for row in msd_ds]
corpus_docs = imdb_docs + msd_docs
corpus_df = pd.DataFrame(corpus_docs)

print(f"IMDb documents: {len(imdb_docs)}")
print(f"Million Song documents: {len(msd_docs)}")
print(f"Total documents: {len(corpus_df)}")
print(corpus_df[["source", "title"]].head())

## 4. Build the Retrieval Index

The notebook uses a dense embedding model plus FAISS cosine search. The cache is optional but helps a lot when you rerun the notebook in Colab.

In [ ]:
import time

import faiss
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CACHE_STEM = f"imdb_{IMDB_LIMIT}_msd_{MSD_LIMIT}"
DOCS_CACHE = WORK_DIR / f"{CACHE_STEM}_docs.pkl"
EMB_CACHE = WORK_DIR / f"{CACHE_STEM}_embeddings.npy"
INDEX_CACHE = WORK_DIR / f"{CACHE_STEM}_faiss.index"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Embedding device: {DEVICE}")
embedding_model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

if DOCS_CACHE.exists() and EMB_CACHE.exists() and INDEX_CACHE.exists():
    corpus_df = pd.read_pickle(DOCS_CACHE)
    embeddings = np.load(EMB_CACHE)
    index = faiss.read_index(str(INDEX_CACHE))
    print(f"Loaded cached corpus and index from {WORK_DIR}")
else:
    texts = corpus_df["text"].tolist()
    print(f"Encoding {len(texts)} documents with {EMBED_MODEL_NAME}...")
    embeddings = embedding_model.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    corpus_df.to_pickle(DOCS_CACHE)
    np.save(EMB_CACHE, embeddings)
    faiss.write_index(index, str(INDEX_CACHE))
    print(f"Built and cached index with {index.ntotal} vectors")


def retrieve_top_k(query: str, top_k: int = TOP_K) -> pd.DataFrame:
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")
    scores, indices = index.search(query_embedding, top_k)
    rows = corpus_df.iloc[indices[0]].copy()
    rows = rows.reset_index(drop=True)
    rows["score"] = scores[0]
    return rows


print(corpus_df[["source", "title"]].head())
print(f"Index size: {index.ntotal}")

## 5. Load Llama 3.2 3B Instruct

This model is gated on Hugging Face. Accept the model license on your account and provide a token if Colab does not already have one.

In [ ]:
import gc
import getpass

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLAMA_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

if "llm_model" in globals():
    del llm_model
if "llm_tokenizer" in globals():
    del llm_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

hf_token = HF_TOKEN
if not hf_token:
    try:
        from google.colab import userdata

        hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or userdata.get("HUGGING_FACE_HUB_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token for Llama 3.2 3B: ").strip()

if not hf_token:
    raise ValueError("A Hugging Face token is required to load meta-llama/Llama-3.2-3B-Instruct.")

print(f"Loading tokenizer: {LLAMA_MODEL_ID}")
llm_tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID, token=hf_token)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print(f"Loading model: {LLAMA_MODEL_ID}")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_ID,
    token=hf_token,
    device_map="auto",
    quantization_config=quant_config,
)
llm_model.eval()
print("Llama 3.2 3B ready")

## 6. RAG Answer Helper

This helper retrieves the top-k documents, builds a compact evidence block, and asks Llama 3.2 to answer only from that context.

In [ ]:
def build_context(rows: pd.DataFrame, max_chars: int = 3200) -> str:
    blocks = []
    used = 0
    for _, row in rows.iterrows():
        source = row.get("source", "")
        title = row.get("title", "")
        score = row.get("score", 0.0)
        text = normalize_text(row.get("text", ""))
        block = f"[source={source} | title={title} | score={score:.3f}] {text}"
        if used + len(block) > max_chars:
            block = block[: max(0, max_chars - used)]
        if block.strip():
            blocks.append(block)
            used += len(block)
        if used >= max_chars:
            break
    return "\n\n".join(blocks)


def build_prompt(question: str, evidence: str) -> str:
    return f"""You are a careful question-answering assistant.
Use only the evidence below.
If the evidence does not contain enough information, say so clearly.
Keep the answer short and directly relevant.

Evidence:
{evidence}

Question: {question}
Answer:"""


def answer_question(question: str, top_k: int = TOP_K, max_new_tokens: int = 180) -> dict:
    retrieved = retrieve_top_k(question, top_k=top_k)
    evidence = build_context(retrieved)
    prompt = build_prompt(question, evidence)

    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {key: value.to(llm_model.device) for key, value in inputs.items()}

    with torch.inference_mode():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return {
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
        "evidence": evidence,
    }


def show_answer(result: dict) -> None:
    print("Question:")
    print(result["question"])
    print("\nRetrieved documents:")
    for idx, row in result["retrieved"].iterrows():
        print(f"{idx + 1}. [{row['source']}] {row['title']} | score={row['score']:.3f}")
    print("\nAnswer:")
    print(result["answer"])


## 7. Example Questions

Run these after the index and model cells finish. Replace them with your own questions once you are ready.

In [ ]:
example_questions = [
    "What do the IMDb reviews suggest about the movie's overall reception?",
    "Which retrieved documents mention a song title or artist name most clearly?",
    "What themes or descriptions are repeated in the IMDb reviews?",
]

for question in example_questions:
    print("=" * 88)
    result = answer_question(question)
    show_answer(result)
    print()